# Day 17 — Hybrid Fraud Detection and Risk Scoring

This notebook combines:

- Random Forest fraud probability: **50%**
- Isolation Forest anomaly score: **25%**
- Autoencoder reconstruction score: **25%**

It uses the balanced training dataset and the unchanged validation dataset. The test dataset is intentionally not used.


In [ ]:
%matplotlib inline

import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")
print("TensorFlow version:", tf.__version__)


## 1. Load prepared datasets

The fallback filenames allow the notebook to work with the names previously used in this project.


In [ ]:
def find_existing_file(file_names):
    for file_name in file_names:
        if Path(file_name).exists():
            return file_name
    raise FileNotFoundError(
        "None of these files were found: " + ", ".join(file_names)
    )


train_file = find_existing_file([
    "train_fraud_balanced.csv",
    "train_fraud_balanced .csv"
])

validation_file = find_existing_file([
    "validation_fraud_dataset.csv",
    "validation_fraud.csv"
])

train_df = pd.read_csv(train_file)
validation_df = pd.read_csv(validation_file)

print("Training file:", train_file)
print("Validation file:", validation_file)
print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)


In [ ]:
target_column = "is_fraud"

if target_column not in train_df.columns:
    raise ValueError("is_fraud is missing from training data.")

if target_column not in validation_df.columns:
    raise ValueError("is_fraud is missing from validation data.")

print("Training target distribution:")
print(train_df[target_column].value_counts())

print()
print("Validation target distribution:")
print(validation_df[target_column].value_counts())


## 2. Prepare common numeric features

Only training statistics are used for missing-value handling. This prevents validation leakage.


In [ ]:
X_train = train_df.drop(columns=[target_column]).copy()
y_train = pd.to_numeric(train_df[target_column], errors="coerce")

X_validation = validation_df.drop(columns=[target_column]).copy()
y_validation = pd.to_numeric(validation_df[target_column], errors="coerce")

common_columns = [
    column for column in X_train.columns
    if column in X_validation.columns
]

X_train = X_train[common_columns].copy()
X_validation = X_validation[common_columns].copy()

for column in common_columns:
    X_train[column] = pd.to_numeric(X_train[column], errors="coerce")
    X_validation[column] = pd.to_numeric(X_validation[column], errors="coerce")

usable_columns = [
    column for column in X_train.columns
    if X_train[column].notna().any()
]

if not usable_columns:
    raise ValueError("No usable numeric features were found.")

X_train = X_train[usable_columns].copy()
X_validation = X_validation[usable_columns].copy()

print("Usable numeric features:", len(usable_columns))
print(usable_columns)


In [ ]:
valid_train_rows = y_train.notna()
valid_validation_rows = y_validation.notna()

X_train = X_train.loc[valid_train_rows].reset_index(drop=True)
y_train = y_train.loc[valid_train_rows].astype(int).reset_index(drop=True)

X_validation = X_validation.loc[valid_validation_rows].reset_index(drop=True)
y_validation = y_validation.loc[valid_validation_rows].astype(int).reset_index(drop=True)

training_medians = X_train.median()

X_train = X_train.fillna(training_medians).fillna(0)
X_validation = X_validation.fillna(training_medians).fillna(0)

print("Training missing values:", X_train.isnull().sum().sum())
print("Validation missing values:", X_validation.isnull().sum().sum())
print("Prepared training shape:", X_train.shape)
print("Prepared validation shape:", X_validation.shape)


## 3. Random Forest supervised score

Random Forest uses all balanced training rows.


In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=1
)

random_forest_model.fit(X_train, y_train)

rf_fraud_scores = random_forest_model.predict_proba(
    X_validation
)[:, 1]

print("Random Forest trained successfully!")
print("Probability range:", rf_fraud_scores.min(), "to", rf_fraud_scores.max())


## 4. Normal-only data for anomaly models

Isolation Forest and Autoencoder learn only normal transaction behaviour.


In [ ]:
X_train_normal = X_train.loc[y_train == 0].copy()

if X_train_normal.empty:
    raise ValueError("No normal training transactions were found.")

anomaly_feature_scaler = StandardScaler()

X_train_normal_scaled = anomaly_feature_scaler.fit_transform(
    X_train_normal
).astype("float32")

X_validation_scaled = anomaly_feature_scaler.transform(
    X_validation
).astype("float32")

print("Normal training rows:", X_train_normal_scaled.shape)
print("Validation rows:", X_validation_scaled.shape)


## 5. Isolation Forest anomaly score


In [ ]:
isolation_model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42,
    n_jobs=1
)

isolation_model.fit(X_train_normal_scaled)

normal_isolation_raw = -isolation_model.decision_function(
    X_train_normal_scaled
).reshape(-1, 1)

validation_isolation_raw = -isolation_model.decision_function(
    X_validation_scaled
).reshape(-1, 1)

isolation_score_scaler = MinMaxScaler()
isolation_score_scaler.fit(normal_isolation_raw)

isolation_scores = isolation_score_scaler.transform(
    validation_isolation_raw
).ravel()

isolation_scores = np.clip(isolation_scores, 0, 1)

print("Isolation Forest trained successfully!")
print("Score range:", isolation_scores.min(), "to", isolation_scores.max())


## 6. Autoencoder reconstruction score


In [ ]:
number_of_features = X_train_normal_scaled.shape[1]
encoding_size = max(2, number_of_features // 2)

input_layer = Input(shape=(number_of_features,))
encoded = Dense(encoding_size, activation="relu")(input_layer)
bottleneck = Dense(max(2, encoding_size // 2), activation="relu")(encoded)
decoded = Dense(encoding_size, activation="relu")(bottleneck)
output_layer = Dense(number_of_features, activation="linear")(decoded)

autoencoder_model = Model(
    inputs=input_layer,
    outputs=output_layer
)

autoencoder_model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

autoencoder_model.summary()


In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

autoencoder_history = autoencoder_model.fit(
    X_train_normal_scaled,
    X_train_normal_scaled,
    epochs=50,
    batch_size=128,
    validation_split=0.20,
    callbacks=[early_stopping],
    shuffle=True,
    verbose=1
)

print("Autoencoder trained successfully!")


In [ ]:
normal_reconstructed = autoencoder_model.predict(
    X_train_normal_scaled,
    verbose=0
)

normal_reconstruction_errors = np.mean(
    np.square(X_train_normal_scaled - normal_reconstructed),
    axis=1
).reshape(-1, 1)

validation_reconstructed = autoencoder_model.predict(
    X_validation_scaled,
    verbose=0
)

validation_reconstruction_errors = np.mean(
    np.square(X_validation_scaled - validation_reconstructed),
    axis=1
).reshape(-1, 1)

autoencoder_score_scaler = MinMaxScaler()
autoencoder_score_scaler.fit(normal_reconstruction_errors)

autoencoder_scores = autoencoder_score_scaler.transform(
    validation_reconstruction_errors
).ravel()

autoencoder_scores = np.clip(autoencoder_scores, 0, 1)

print("Autoencoder scores created successfully!")
print("Score range:", autoencoder_scores.min(), "to", autoencoder_scores.max())


## 7. Threshold helper and individual validation metrics

Thresholds are selected using validation F1 score. The test dataset remains untouched.


In [ ]:
def find_best_threshold(y_true, scores):
    precision_values, recall_values, thresholds = precision_recall_curve(
        y_true,
        scores
    )

    if len(thresholds) == 0:
        return 0.5

    f1_values = (
        2 * precision_values[:-1] * recall_values[:-1]
    ) / (
        precision_values[:-1] + recall_values[:-1] + 1e-10
    )

    return float(thresholds[np.argmax(f1_values)])


def calculate_metrics(model_name, y_true, scores):
    threshold = find_best_threshold(y_true, scores)
    predictions = (scores >= threshold).astype(int)

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1_score": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, scores),
        "threshold": threshold
    }

    return metrics, predictions


In [ ]:
rf_metrics, rf_predictions = calculate_metrics(
    "Random Forest",
    y_validation,
    rf_fraud_scores
)

isolation_metrics, isolation_predictions = calculate_metrics(
    "Isolation Forest",
    y_validation,
    isolation_scores
)

autoencoder_metrics, autoencoder_predictions = calculate_metrics(
    "Autoencoder",
    y_validation,
    autoencoder_scores
)

print("Individual model validation metrics created!")


## 8. Hybrid fraud risk score


In [ ]:
hybrid_scores = (
    0.50 * rf_fraud_scores
    + 0.25 * isolation_scores
    + 0.25 * autoencoder_scores
)

hybrid_metrics, hybrid_predictions = calculate_metrics(
    "Hybrid Model",
    y_validation,
    hybrid_scores
)

hybrid_threshold = hybrid_metrics["threshold"]
risk_scores = np.round(hybrid_scores * 100, 2)

print("Hybrid threshold:", round(hybrid_threshold, 4))
print("Equivalent risk score:", round(hybrid_threshold * 100, 2))
print("Risk score range:", risk_scores.min(), "to", risk_scores.max())


In [ ]:
def assign_risk_level(score):
    if score < 30:
        return "Low"
    if score < 60:
        return "Medium"
    if score < 80:
        return "High"
    return "Critical"


risk_levels = [assign_risk_level(score) for score in risk_scores]

print("Risk distribution:")
print(pd.Series(risk_levels).value_counts())


## 9. Final results and evaluation


In [ ]:
hybrid_results = pd.DataFrame({
    "validation_row": np.arange(len(y_validation)),
    "actual_fraud": y_validation.values,
    "random_forest_score": rf_fraud_scores,
    "isolation_anomaly_score": isolation_scores,
    "autoencoder_reconstruction_score": autoencoder_scores,
    "hybrid_score": hybrid_scores,
    "risk_score": risk_scores,
    "risk_level": risk_levels,
    "predicted_fraud": hybrid_predictions
})

if "transaction_id" in validation_df.columns:
    transaction_ids = validation_df.loc[
        valid_validation_rows,
        "transaction_id"
    ].reset_index(drop=True)
    hybrid_results.insert(1, "transaction_id", transaction_ids)

hybrid_results.head()


In [ ]:
print("HYBRID FRAUD DETECTION RESULTS")
print("-" * 35)
print("Accuracy:", round(hybrid_metrics["accuracy"], 4))
print("Precision:", round(hybrid_metrics["precision"], 4))
print("Recall:", round(hybrid_metrics["recall"], 4))
print("F1 Score:", round(hybrid_metrics["f1_score"], 4))
print("ROC-AUC:", round(hybrid_metrics["roc_auc"], 4))

print()
print("Classification report:")
print(classification_report(
    y_validation,
    hybrid_predictions,
    labels=[0, 1],
    target_names=["Normal", "Fraud"],
    zero_division=0
))


In [ ]:
hybrid_confusion = confusion_matrix(
    y_validation,
    hybrid_predictions,
    labels=[0, 1]
)

plt.figure(figsize=(6, 4))
sns.heatmap(
    hybrid_confusion,
    annot=True,
    fmt="d",
    cmap="Reds",
    xticklabels=["Normal", "Fraud"],
    yticklabels=["Normal", "Fraud"]
)
plt.title("Hybrid Fraud Detection Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()


In [ ]:
risk_order = ["Low", "Medium", "High", "Critical"]

plt.figure(figsize=(8, 5))
sns.countplot(
    data=hybrid_results,
    x="risk_level",
    order=risk_order,
    color="steelblue"
)
plt.title("Fraud Risk Level Distribution")
plt.xlabel("Risk Level")
plt.ylabel("Transactions")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(
    data=hybrid_results,
    x="risk_score",
    hue="actual_fraud",
    bins=40,
    element="step"
)
plt.axvline(
    hybrid_threshold * 100,
    color="black",
    linestyle="--",
    label="Selected fraud threshold"
)
plt.title("Hybrid Risk Score Distribution")
plt.xlabel("Risk Score (0–100)")
plt.ylabel("Transactions")
plt.tight_layout()
plt.show()


In [ ]:
high_risk_transactions = hybrid_results[
    hybrid_results["risk_level"].isin(["High", "Critical"])
].sort_values(
    "risk_score",
    ascending=False
)

print("High/Critical risk transactions:", len(high_risk_transactions))
high_risk_transactions.head(20)


In [ ]:
risk_summary = (
    hybrid_results
    .groupby("risk_level", observed=False)["actual_fraud"]
    .agg(["count", "sum", "mean"])
    .reindex(["Low", "Medium", "High", "Critical"], fill_value=0)
    .reset_index()
)

risk_summary["mean"] = risk_summary["mean"] * 100
risk_summary = risk_summary.rename(columns={
    "count": "transactions",
    "sum": "fraud_transactions",
    "mean": "fraud_rate_percent"
})

risk_summary


## 10. Model comparison and saved files


In [ ]:
model_comparison = pd.DataFrame([
    rf_metrics,
    isolation_metrics,
    autoencoder_metrics,
    hybrid_metrics
]).sort_values(
    "f1_score",
    ascending=False
).reset_index(drop=True)

model_comparison.round(4)


In [ ]:
hybrid_results.to_csv(
    "day17_hybrid_fraud_risk_results.csv",
    index=False
)

high_risk_transactions.to_csv(
    "day17_high_risk_transactions.csv",
    index=False
)

model_comparison.to_csv(
    "day17_model_comparison.csv",
    index=False
)

risk_summary.to_csv(
    "day17_risk_summary.csv",
    index=False
)

print("Files saved successfully:")
print("1. day17_hybrid_fraud_risk_results.csv")
print("2. day17_high_risk_transactions.csv")
print("3. day17_model_comparison.csv")
print("4. day17_risk_summary.csv")


## Execution note

Run every cell in order. Day 17 uses only the prepared training and validation files. Keep the test dataset untouched until the final selected model is evaluated.
